# Crop Recommendation — Reference Solution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/Sandbox_Training/blob/main/notebooks/crop-recommendation-reference/notebook/Crop_Recommendation_Reference.ipynb)

Reference solution for the **Crop Recommendation** dataset in the ITU AI for Good sandbox training. It trains and compares a **Logistic Regression** and a **Decision Tree** classifier.

Click the **Open in Colab** badge above to run this notebook end-to-end in Google Colab — no setup or download required. The dataset is loaded straight from the public catalog.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

RANDOM_STATE = 42

## 1. Load the data

Loaded directly from the public GitHub catalog — no local file needed.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/CrashingGuru/Sandbox_Training/main/"
    "datasets/crop-recommendation/data/Crop_recommendation.csv"
)
df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
df.head()

## 2. Quick look

The target column is `label` (the recommended crop). The 22 classes are balanced.

In [ ]:
print("Classes:", df['label'].nunique())
print(df['label'].value_counts().head())
df.describe()

## 3. Train / test split

Stratified so every crop is represented in both splits.

In [ ]:
X = df.drop(columns='label')
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(X_train.shape, X_test.shape)

## 4. Logistic Regression (with scaling)

Linear models benefit from standardized features, so we scale first.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)
logreg_acc = accuracy_score(y_test, logreg.predict(X_test_s))
print(f"Logistic Regression accuracy: {logreg_acc:.3f}")

## 5. Decision Tree (no scaling needed)

Trees are scale-invariant, so we fit on the raw features.

In [ ]:
tree = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
tree.fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
print(f"Decision Tree (depth 6) accuracy: {tree_acc:.3f}")

## 6. Compare & report

Per-class precision/recall for the stronger model.

In [ ]:
print(f"Logistic Regression: {logreg_acc:.3f}")
print(f"Decision Tree:       {tree_acc:.3f}")

if logreg_acc >= tree_acc:
    best_name, best_pred = "Logistic Regression", logreg.predict(X_test_s)
else:
    best_name, best_pred = "Decision Tree", tree.predict(X_test)
print(f"\nClassification report — {best_name}:\n")
print(classification_report(y_test, best_pred))

## 7. Feature importance (Decision Tree)

Which measurements drive the recommendation.

In [ ]:
importances = pd.Series(tree.feature_importances_, index=X.columns).sort_values()
importances.plot.barh()
plt.title("Decision Tree feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()